# 01 · Ingest open data: ACS climate projections + ABS population

| Source | Publisher | What | Licence |
|---|---|---|---|
| **BARPA-R** (AWS Open Data `s3://bom-opendata-climate`) | Bureau of Meteorology, for the Australian Climate Service | Downscaled CMIP6 projections, 0.2° grid, monthly NetCDF | CC BY 4.0 |
| **ASGS 2021** SA2 + State boundaries | Australian Bureau of Statistics | Shapefiles (GDA2020) | CC BY 4.0 |
| **Regional population 2024-25** | Australian Bureau of Statistics | Estimated Resident Population by SA2, 2001-2025 (Excel) | CC BY 4.0 |

**accessing large datasets and querying APIs:** we *discover* files with the S3 ListObjects REST API
instead of hard-coding them, then land them once in a governed Unity Catalog Volume.

In [ ]:
%run ./00_config

## 1 · Query the S3 API to discover which BARPA files exist

In [ ]:
import re, requests, xml.etree.ElementTree as ET

def list_barpa(scenario: str, var: str) -> list[dict]:
    """Call the public S3 ListObjectsV2 API (no credentials) for one scenario/variable."""
    prefix = BARPA_PREFIX.format(scenario=scenario, var=var)
    xml = requests.get(BARPA_BUCKET, params={"list-type": "2", "prefix": prefix}, timeout=60).text
    ns = {"s3": "http://s3.amazonaws.com/doc/2006-03-01/"}
    out = []
    for c in ET.fromstring(xml).findall("s3:Contents", ns):
        key = c.find("s3:Key", ns).text
        decade = re.search(r"_mon_(\d{6}-\d{6})\.nc$", key).group(1)
        out.append({"scenario": scenario, "variable": var, "decade": decade, "key": key,
                    "bytes": int(c.find("s3:Size", ns).text)})
    return out

catalogue = [r for sc in ["historical", *FUTURES] for v in VARIABLES for r in list_barpa(sc, v)]
cat_df = spark.createDataFrame(catalogue)
print(f"{len(catalogue)} monthly files available across {cat_df.select('scenario').distinct().count()} scenarios")
display(cat_df.groupBy("scenario", "variable").agg({"decade": "count", "bytes": "sum"}).orderBy("scenario", "variable"))

## 2 · Select the decades we need and land them in the Volume (idempotent)

In [ ]:
import os, shutil, tempfile
from concurrent.futures import ThreadPoolExecutor

wanted = {BASELINE} | {(sc, d) for sc, ds in FUTURES.items() for d in ds}
selected = [r for r in catalogue if (r["scenario"], r["decade"]) in wanted]

def download(url: str, dest: str, expected: int | None = None) -> dict:
    """Stream to local disk, then copy to the Volume. Skips files already landed with the right size."""
    if os.path.exists(dest) and (expected is None or os.path.getsize(dest) == expected):
        return {"url": url, "path": dest, "bytes": os.path.getsize(dest), "status": "cached"}
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    with tempfile.NamedTemporaryFile(delete=False) as tmp:
        with requests.get(url, stream=True, timeout=300, headers={"User-Agent": "Mozilla/5.0"}) as r:
            r.raise_for_status()
            for chunk in r.iter_content(8 << 20):
                tmp.write(chunk)
    shutil.copy(tmp.name, dest); os.remove(tmp.name)
    return {"url": url, "path": dest, "bytes": os.path.getsize(dest), "status": "downloaded"}

jobs = [(f"{BARPA_BUCKET}/{r['key']}", f"{VOL}/barpa/{os.path.basename(r['key'])}", r["bytes"]) for r in selected]
jobs += [(url, f"{VOL}/abs/{name}", None) for name, url in ABS_FILES.items()]

with ThreadPoolExecutor(8) as pool:
    results = list(pool.map(lambda j: download(*j), jobs))

for r in results:
    print(f"{r['status']:>10}  {r['bytes']/1e6:8.1f} MB  {os.path.basename(r['path'])}")

## 3 · Record a file manifest (provenance for lineage + attribution)

In [ ]:
from pyspark.sql import functions as F

manifest = (spark.createDataFrame(results)
            .withColumn("publisher", F.when(F.col("url").contains("bom-opendata"), "Bureau of Meteorology / Australian Climate Service")
                                      .otherwise("Australian Bureau of Statistics"))
            .withColumn("licence", F.lit("CC BY 4.0"))
            .withColumn("landed_at", F.current_timestamp()))
manifest.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQ}.bronze_file_manifest")
spark.sql(f"COMMENT ON TABLE {FQ}.bronze_file_manifest IS 'Source files landed from AWS Open Data (BARPA-R, CC BY 4.0) and ABS (CC BY 4.0)'")
display(spark.table(f"{FQ}.bronze_file_manifest").select("publisher", "path", "bytes", "status"))
print(f"Total landed: {sum(r['bytes'] for r in results)/1e9:.2f} GB in {len(results)} files")